# YOLO26 Semantic Segmentation (`seg-yolov26-semantic`)

## 1. Setup & imports

In [ ]:
from __future__ import annotations
import os, re, glob, json, math, time, random, shutil, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F

def _pip(pkg):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=True)

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
except ModuleNotFoundError:
    _pip("albumentations"); import albumentations as A
    from albumentations.pytorch import ToTensorV2
try:
    from ultralytics import YOLO
    import ultralytics
except ModuleNotFoundError:
    _pip("ultralytics"); from ultralytics import YOLO
    import ultralytics
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(x, **k): return x

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__, "| ultralytics:", ultralytics.__version__, "| albumentations:", A.__version__)
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only")

### Reproducibility 

In [ ]:
SEED = 42
def set_seed(seed: int = SEED) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(); print("seed set to", SEED)

## 2. CONFIG 

In [ ]:
def find_data_root() -> str:
    candidates = [
        "/kaggle/input/datasets/samirhossain2001/automatic-hair-removal/Dataset",
        "e:/CSE438_DIP/CSE438_Assignment/Dataset",
    ]
    candidates += [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/black_masks", recursive=True)]
    for root in candidates:
        if root and all(os.path.isdir(os.path.join(root, d)) for d in ("images", "black_masks", "white_masks")):
            return root
    raise FileNotFoundError("No folder with images/ black_masks/ white_masks/ found.")

DATA_ROOT = find_data_root()
CONFIG = {
    "DATA_ROOT":    DATA_ROOT,
    "IMAGES_DIR":   os.path.join(DATA_ROOT, "images"),
    "BLACK_DIR":    os.path.join(DATA_ROOT, "black_masks"),
    "WHITE_DIR":    os.path.join(DATA_ROOT, "white_masks"),
    "OUT_DIR":      "/kaggle/working" if os.path.isdir("/kaggle/working") else ".",
    "YOLO_DS":      os.path.join("/kaggle/working" if os.path.isdir("/kaggle/working") else ".", "yolo_ds"),
    "MODEL_NAME":   "yolo26_sem",
    "CHECKPOINT":   "yolo26n-sem.pt",
    "SEED":         42,
    "IMG_SIZE":     512,
    "NUM_CLASSES":  3,
    "CLASS_NAMES":  {0: "background", 1: "dark_hair_ruler", 2: "light_hair_uv"},
    "CLASS_COLORS": {1: (255, 0, 0), 2: (0, 255, 0)},
    "MASK_THRESH":  127,
    "SPLIT_RATIO":  (0.70, 0.15, 0.15),
    "EPOCHS":       80,
    "BATCH_SIZE":   8,
    "YOLO_TRAIN": {
        "optimizer": "AdamW", "cos_lr": True, "warmup_epochs": 3,     
        "fliplr": 0.5, "flipud": 0.5, "degrees": 20.0,                
        "scale": 0.1, "translate": 0.05,                             
        "hsv_h": 0.015, "hsv_s": 0.3, "hsv_v": 0.3,                  
        "mosaic": 0.0, "mixup": 0.0, "copy_paste": 0.0,             
    },
    "PARAMS_M":     {"deeplabv3_resnet50": 42.0, "segformer_b0": 3.7, "yolo26_sem": 1.6},
}
for k, v in CONFIG.items(): print(f"{k:12}: {v}")

## 3. Shared utilities

In [ ]:
def group_key(filename: str) -> str:
    stem = filename[:-4] if filename.lower().endswith(".png") else filename
    stem = re.sub(r"^crop_\d+_", "", stem); stem = re.sub(r"_crop_\d+$", "", stem)
    return stem

def load_rgb(filename: str, size: int | None = None) -> np.ndarray:
    size = size or CONFIG["IMG_SIZE"]
    return np.asarray(Image.open(os.path.join(CONFIG["IMAGES_DIR"], filename)).convert("RGB").resize((size, size)))

def _mask(folder: str, filename: str, size: int) -> np.ndarray:
    m = Image.open(os.path.join(folder, filename)).convert("L").resize((size, size), Image.NEAREST)
    return np.asarray(m) > CONFIG["MASK_THRESH"]

def build_label(filename: str, size: int | None = None) -> np.ndarray:
    size = size or CONFIG["IMG_SIZE"]
    label = np.zeros((size, size), np.uint8)
    if filename in BLACK_MASKS: label[_mask(CONFIG["BLACK_DIR"], filename, size)] = 1
    if filename in WHITE_MASKS: label[_mask(CONFIG["WHITE_DIR"], filename, size)] = 2
    return label

def overlay(image: np.ndarray, label: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    out = image.astype(np.float32).copy()
    for cls, colour in CONFIG["CLASS_COLORS"].items():
        m = label == cls
        for c in range(3): out[..., c][m] = (1 - alpha) * out[..., c][m] + alpha * colour[c]
    return out.astype(np.uint8)

BLACK_MASKS = set(os.listdir(CONFIG["BLACK_DIR"]))
IMAGE_FILES = sorted(os.listdir(CONFIG["IMAGES_DIR"]))
WHITE_MASKS = {f for f in os.listdir(CONFIG["WHITE_DIR"]) if f in set(IMAGE_FILES)}
print("images:", len(IMAGE_FILES))

### Metrics 

In [ ]:
K = CONFIG["NUM_CLASSES"]
class ConfusionMatrix:
    def __init__(self, num_classes: int = K):
        self.k = num_classes
        self.mat = np.zeros((num_classes, num_classes), dtype=np.int64)
    def update(self, pred: np.ndarray, target: np.ndarray) -> None:
        idx = target.reshape(-1).astype(np.int64) * self.k + pred.reshape(-1).astype(np.int64)
        self.mat += np.bincount(idx, minlength=self.k ** 2).reshape(self.k, self.k)
    def _t(self):
        cm = self.mat.astype(np.float64); tp = np.diag(cm)
        return tp, cm.sum(0) - tp, cm.sum(1) - tp, cm
    @property
    def per_class_iou(self):
        tp, fp, fn, _ = self._t(); return tp / np.clip(tp + fp + fn, 1e-9, None)
    @property
    def per_class_dice(self):
        tp, fp, fn, _ = self._t(); return 2 * tp / np.clip(2 * tp + fp + fn, 1e-9, None)
    @property
    def mean_iou(self): return float(self.per_class_iou.mean())
    @property
    def foreground_mean_iou(self): return float(self.per_class_iou[1:].mean())
    @property
    def mean_dice(self): return float(self.per_class_dice.mean())
    @property
    def pixel_accuracy(self):
        tp, _, _, cm = self._t(); return float(tp.sum() / max(cm.sum(), 1e-9))
    @property
    def mean_pixel_accuracy(self):
        tp, _, _, cm = self._t(); return float((tp / np.clip(cm.sum(1), 1e-9, None)).mean())
    @property
    def matrix(self): return self.mat

def plot_grid(columns, row_labels, suptitle, col_w=2.6, row_h=2.6):
    nrows, ncols = len(row_labels), len(columns)
    fig, ax = plt.subplots(nrows, ncols, figsize=(col_w * ncols, row_h * nrows), squeeze=False)
    for j, (cap, imgs) in enumerate(columns):
        for i, im in enumerate(imgs):
            ax[i, j].imshow(im); ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
            if i == 0: ax[i, j].set_title(cap, fontsize=8)
    for i, lab in enumerate(row_labels): ax[i, 0].set_ylabel(lab, fontsize=9)
    fig.suptitle(suptitle, y=1.01); fig.tight_layout(); plt.show()

IMAGENET_MEAN, IMAGENET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
eval_tf = A.Compose([A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])   
print("ConfusionMatrix + plot_grid ready")

## 4. Load the shared split from NB0

In [ ]:
def find_artifact(name: str):
    for base in ["/kaggle/working", "."] + glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/**/", recursive=True):
        p = os.path.join(base, name)
        if os.path.isfile(p): return p
    return None

def make_grouped_split(files, seed=SEED, ratio=(0.70, 0.15, 0.15)):
    groups = defaultdict(list)
    for f in files: groups[group_key(f)].append(f)
    sized = [(g, len(m)) for g, m in groups.items()]; random.Random(seed).shuffle(sized)
    total = len(files); target = dict(zip(("train", "val", "test"), (r * total for r in ratio)))
    filled = {"train": 0, "val": 0, "test": 0}; assign = {}
    for g, size in sorted(sized, key=lambda gs: -gs[1]):
        sp = max(target, key=lambda k: target[k] - filled[k]); assign[g] = sp; filled[sp] += size
    split = {"train": [], "val": [], "test": []}
    for g in sorted(groups): split[assign[g]].extend(sorted(groups[g]))
    return split

sp_path = find_artifact("split.json")
if sp_path:
    split = {k: json.load(open(sp_path))[k] for k in ("train", "val", "test")}; print("loaded split.json from", sp_path)
else:
    warnings.warn("split.json not found - regenerating identical split (seed 42)."); split = make_grouped_split(IMAGE_FILES)
print({k: len(v) for k, v in split.items()})

## 5. YOLO26-sem (build the semantic dataset)

In [ ]:
import yaml
def build_yolo_dataset():
    root = CONFIG["YOLO_DS"]
    if os.path.isdir(root): shutil.rmtree(root)
    for sp, files in split.items():
        img_dir = os.path.join(root, "images", sp); msk_dir = os.path.join(root, "masks", sp)
        os.makedirs(img_dir, exist_ok=True); os.makedirs(msk_dir, exist_ok=True)
        for fn in files:
            shutil.copy(os.path.join(CONFIG["IMAGES_DIR"], fn), os.path.join(img_dir, fn))
            Image.fromarray(build_label(fn), mode="L").save(os.path.join(msk_dir, fn))   # class-ID PNG
    data = {"path": root, "train": "images/train", "val": "images/val", "test": "images/test",
            "masks_dir": "masks", "names": {int(k): v for k, v in CONFIG["CLASS_NAMES"].items()}}
    yaml_path = os.path.join(root, "data.yaml")
    with open(yaml_path, "w") as f: yaml.safe_dump(data, f, sort_keys=False)
    return yaml_path

DATA_YAML = build_yolo_dataset()
print("wrote", DATA_YAML)
print(open(DATA_YAML).read())

## 6. Train YOLO26-sem

In [ ]:
ymodel = YOLO(CONFIG["CHECKPOINT"])                      
yolo_params_m = sum(p.numel() for p in ymodel.model.parameters()) / 1e6
CONFIG["PARAMS_M"]["yolo26_sem"] = round(yolo_params_m, 2)
print(f"{CONFIG['CHECKPOINT']} | params: {yolo_params_m:.2f}M")

t0 = time.time()
ymodel.train(data=DATA_YAML, epochs=CONFIG["EPOCHS"], imgsz=CONFIG["IMG_SIZE"],
             batch=CONFIG["BATCH_SIZE"], seed=CONFIG["SEED"], deterministic=True,
             project=os.path.join(CONFIG["OUT_DIR"], "yolo_runs"), name="uvfd_sem", exist_ok=True,
             **CONFIG["YOLO_TRAIN"])          
yolo_train_minutes = (time.time() - t0) / 60
yolo_best = os.path.join(CONFIG["OUT_DIR"], "yolo_runs", "uvfd_sem", "weights", "best.pt")
print(f"training done in {yolo_train_minutes:.1f} min | best weights: {yolo_best}")

### training curves


In [ ]:
run_dir = os.path.dirname(os.path.dirname(yolo_best)) 
csv_path = os.path.join(run_dir, "results.csv")
hist = pd.read_csv(csv_path); hist.columns = hist.columns.str.strip()
xcol = "epoch" if "epoch" in hist.columns else hist.columns[0]
loss_cols  = [c for c in hist.columns if "loss" in c.lower()]
train_loss = hist[[c for c in loss_cols if "train" in c.lower()]].sum(axis=1)
val_loss   = hist[[c for c in loss_cols if "val" in c.lower()]].sum(axis=1)
miou_col   = next((c for c in hist.columns if "miou" in c.lower()), None)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist[xcol], train_loss, label="train")
ax[0].plot(hist[xcol], val_loss,   label="val")
ax[0].set_title("YOLO26-sem loss (total = ce+dice+aux)"); ax[0].set_xlabel("epoch"); ax[0].legend()
if miou_col is not None:
    ax[1].plot(hist[xcol], hist[miou_col], color="#4C78A8", label="overall val mIoU (YOLO)")
    ax[1].set_title(f"Validation mIoU (best {hist[miou_col].max():.3f})")
ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()
print("plotted from", csv_path, "| loss columns:", loss_cols, "| mIoU column:", miou_col)

### YOLO prediction helper

In [ ]:
best_model = YOLO(yolo_best) if os.path.isfile(yolo_best) else ymodel

def yolo_predict(filename: str) -> np.ndarray:
    path = os.path.join(CONFIG["IMAGES_DIR"], filename)
    res = best_model.predict(source=path, imgsz=CONFIG["IMG_SIZE"], verbose=False)[0]
    mask = res.semantic_mask.data         
    mask = mask.squeeze().cpu().numpy().astype(np.uint8)
    if mask.shape != (CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]):
        mask = np.asarray(Image.fromarray(mask).resize((CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]), Image.NEAREST))
    return mask

_demo = yolo_predict(split["test"][0]); print("pred shape:", _demo.shape, "| unique:", np.unique(_demo))

## 7. Test-set evaluation

In [ ]:
names = [CONFIG["CLASS_NAMES"][k] for k in range(K)]
yolo_cm = ConfusionMatrix()
for fn in tqdm(split["test"], desc="yolo test eval"):
    yolo_cm.update(yolo_predict(fn), build_label(fn))

per_class = pd.DataFrame({"class": names, "IoU": np.round(yolo_cm.per_class_iou, 4),
                          "Dice": np.round(yolo_cm.per_class_dice, 4)})
print(per_class.to_string(index=False))
print(f"\nmIoU (all 3)      : {yolo_cm.mean_iou:.4f}")
print(f"mIoU (foreground) : {yolo_cm.foreground_mean_iou:.4f}")
print(f"mean Dice (F1)    : {yolo_cm.mean_dice:.4f}")
print(f"pixel accuracy    : {yolo_cm.pixel_accuracy:.4f}")
print(f"mean pixel acc    : {yolo_cm.mean_pixel_accuracy:.4f}")

### Pixel-level confusion matrix 

In [ ]:
cm = yolo_cm.matrix; cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(5.2, 4.4)); im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(K)); ax.set_yticks(range(K)); ax.set_xticklabels(names, rotation=30, ha="right"); ax.set_yticklabels(names)
ax.set_xlabel("predicted"); ax.set_ylabel("ground truth"); ax.set_title("YOLO26-sem confusion (row-normalized)")
for i in range(K):
    for j in range(K): ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", color="white" if cmn[i,j] > 0.5 else "black", fontsize=9)
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

## 8. Error analysis (worst test cases by foreground IoU)

In [ ]:
def yolo_image_fg_iou(fn):
    pred = yolo_predict(fn); c = ConfusionMatrix(); c.update(pred, build_label(fn))
    present = c.matrix.sum(1) > 0; fg = [k for k in (1, 2) if present[k]]
    return (float(c.per_class_iou[fg].mean()) if fg else float(c.per_class_iou[present].mean())), pred

yolo_scored = [(fn, *yolo_image_fg_iou(fn)) for fn in tqdm(split["test"], desc="yolo per-image")]
yolo_scored.sort(key=lambda t: t[1])
def pred_col(cap, fn, pred):
    img = load_rgb(fn); return (cap, [img, overlay(img, build_label(fn)), overlay(img, pred)])
worst = yolo_scored[:5]
print("worst 5:", [(f[:14], round(s, 3)) for f, s, _ in worst])
plot_grid([pred_col(f"{fn[:12]} {s:.2f}", fn, p) for fn, s, p in worst],
          ["image", "truth", "pred"], "YOLO26-sem worst 5 test cases")
off = cm.copy(); np.fill_diagonal(off, 0); i, j = np.unravel_index(off.argmax(), off.shape)
print(f"Most confused: true '{names[i]}' predicted as '{names[j]}' ({100*off[i,j]/cm.sum():.2f}% of pixels)")

## 9. Append YOLO metrics to the shared `results.json`

In [ ]:
res_path = find_artifact("results.json") or os.path.join(CONFIG["OUT_DIR"], "results.json")
results = json.load(open(res_path)) if res_path and os.path.isfile(res_path) else {"models": {}}
results.setdefault("models", {})["yolo26_sem"] = {
    "mIoU": yolo_cm.mean_iou, "foreground_mIoU": yolo_cm.foreground_mean_iou, "mDice": yolo_cm.mean_dice,
    "pixel_acc": yolo_cm.pixel_accuracy, "mean_pixel_acc": yolo_cm.mean_pixel_accuracy,
    "per_class_iou": {names[k]: float(yolo_cm.per_class_iou[k]) for k in range(K)},
    "per_class_dice": {names[k]: float(yolo_cm.per_class_dice[k]) for k in range(K)},
    "params_M": CONFIG["PARAMS_M"]["yolo26_sem"], "epochs_run": CONFIG["EPOCHS"],
    "train_minutes": round(yolo_train_minutes, 1),
}
out_path = os.path.join(CONFIG["OUT_DIR"], "results.json")
with open(out_path, "w") as f: json.dump(results, f, indent=2)
print("saved", out_path); print(json.dumps(results["models"]["yolo26_sem"], indent=2))

---
## 10. Final 3-model comparison

### 10.1 Merge metrics 

In [ ]:
merged = {}
for p in set(glob.glob("/kaggle/input/**/results.json", recursive=True) +
             glob.glob(os.path.join(CONFIG["OUT_DIR"], "results.json"))):
    try: merged.update(json.load(open(p)).get("models", {}))
    except Exception: pass
merged.update(results["models"])   
ORDER = ["deeplabv3_resnet50", "segformer_b0", "yolo26_sem"]
models = [m for m in ORDER if m in merged]
print("models available for comparison:", models)
assert models, "no model metrics found - attach NB1/NB2 outputs (results.json)"

### 10.2 Summary table

In [ ]:
def params_of(m): return merged[m].get("params_M", CONFIG["PARAMS_M"].get(m, float("nan")))
rows = []
for m in models:
    d = merged[m]; pci = d.get("per_class_iou", {})
    rows.append({
        "model": m,
        "foreground_mIoU": round(d.get("foreground_mIoU", float("nan")), 4),
        "mIoU": round(d.get("mIoU", float("nan")), 4),
        "IoU_dark": round(pci.get("dark_hair_ruler", float("nan")), 4),
        "IoU_light": round(pci.get("light_hair_uv", float("nan")), 4),
        "mDice": round(d.get("mDice", float("nan")), 4),
        "pixel_acc": round(d.get("pixel_acc", float("nan")), 4),
        "params_M": params_of(m),
        "train_min": d.get("train_minutes", float("nan")),
    })
comp = pd.DataFrame(rows).set_index("model")
print(comp.to_string())

### 10.3 Metric bar charts overall vs foreground mIoU and per-class IoU

In [ ]:
x = np.arange(len(models)); w = 0.38
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].bar(x - w/2, [merged[m].get("mIoU", np.nan) for m in models], w, label="overall mIoU", color="#4274D9")
ax[0].bar(x + w/2, [merged[m].get("foreground_mIoU", np.nan) for m in models], w, label="foreground mIoU", color="#95CCDD")
ax[0].set_xticks(x); ax[0].set_xticklabels(models, rotation=15); ax[0].set_title("mIoU"); ax[0].legend()
for cls, col, off_ in [("dark_hair_ruler", "#346739", -w/2), ("light_hair_uv", "#79AE6F", w/2)]:
    ax[1].bar(x + off_, [merged[m].get("per_class_iou", {}).get(cls, np.nan) for m in models], w, label=cls, color=col)
ax[1].set_xticks(x); ax[1].set_xticklabels(models, rotation=15); ax[1].set_title("Per-class foreground IoU"); ax[1].legend()
plt.tight_layout(); plt.show()

### 10.4 Radar chart 

In [ ]:
metrics = ["foreground_mIoU", "mIoU", "mDice", "pixel_acc"]
def getm(m, key):
    d = merged[m]
    return d.get(key, np.nan) if key in d else d.get("per_class_iou", {}).get(key, np.nan)
angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist(); angles += angles[:1]
fig = plt.figure(figsize=(5.5, 5.5)); ax = plt.subplot(111, polar=True)
for m in models:
    vals = [getm(m, k) for k in metrics]; vals += vals[:1]
    ax.plot(angles, vals, label=m); ax.fill(angles, vals, alpha=0.08)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(metrics); ax.set_title("Metric profile (higher = better)")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1)); plt.tight_layout(); plt.show()

### 10.5 Accuracy vs cost 

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.2))
for m in models:
    ax.scatter(params_of(m), merged[m].get("foreground_mIoU", np.nan), s=90)
    ax.annotate(m, (params_of(m), merged[m].get("foreground_mIoU", np.nan)),
                textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.set_xlabel("parameters (M)"); ax.set_ylabel("foreground mIoU"); ax.set_title("Accuracy vs model size")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 10.6 Load all three models for qualitative + failure analysis

In [ ]:
predictors = {}

dl_ckpt = find_artifact("deeplabv3_resnet50_best.pt")
if dl_ckpt:
    from torchvision.models.segmentation import deeplabv3_resnet50
    _dl = deeplabv3_resnet50(weights=None, aux_loss=True)
    _dl.classifier[4] = nn.Conv2d(256, K, 1); _dl.aux_classifier[4] = nn.Conv2d(256, K, 1)
    _dl.load_state_dict(torch.load(dl_ckpt, map_location=DEVICE)); _dl.to(DEVICE).eval()
    def _pred_dl(fn):
        x = eval_tf(image=load_rgb(fn))["image"].unsqueeze(0).to(DEVICE)
        with torch.no_grad(): return _dl(x)["out"].argmax(1)[0].cpu().numpy().astype(np.uint8)
    predictors["deeplabv3_resnet50"] = _pred_dl

sf_ckpt = find_artifact("segformer_b0_best.pt")
if sf_ckpt:
    from transformers import SegformerForSemanticSegmentation
    class _SW(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                "nvidia/segformer-b0-finetuned-ade-512-512", num_labels=K, ignore_mismatched_sizes=True)
        def forward(self, x):
            lg = self.net(pixel_values=x).logits
            return F.interpolate(lg, size=(CONFIG["IMG_SIZE"], CONFIG["IMG_SIZE"]), mode="bilinear", align_corners=False)
    _sf = _SW(); _sf.load_state_dict(torch.load(sf_ckpt, map_location=DEVICE)); _sf.to(DEVICE).eval()
    def _pred_sf(fn):
        x = eval_tf(image=load_rgb(fn))["image"].unsqueeze(0).to(DEVICE)
        with torch.no_grad(): return _sf(x).argmax(1)[0].cpu().numpy().astype(np.uint8)
    predictors["segformer_b0"] = _pred_sf

predictors["yolo26_sem"] = yolo_predict
avail = [m for m in ORDER if m in predictors]
print("predictors available:", avail)

### 10.7 ualitative comparison

In [ ]:
n = len(yolo_scored); picks = [yolo_scored[i][0] for i in (0, n//4, n//2, 3*n//4, n-1)]
imgs = {fn: load_rgb(fn) for fn in picks}
columns = [("image", [imgs[fn] for fn in picks]),
           ("ground truth", [overlay(imgs[fn], build_label(fn)) for fn in picks])]
for m in avail:
    columns.append((m, [overlay(imgs[fn], predictors[m](fn)) for fn in picks]))
plot_grid(columns, [fn[:14] for fn in picks],
          "Side-by-side test predictions (rows = images worst->best; red=dark, green=light)")

### 10.8 Cross-model failure analysis

In [ ]:
per_img = {m: [] for m in avail}
test_files = split["test"]
for fn in tqdm(test_files, desc="scoring all models"):
    gt = build_label(fn)
    for m in avail:
        c = ConfusionMatrix(); c.update(predictors[m](fn), gt)
        present = c.matrix.sum(1) > 0; fg = [k for k in (1, 2) if present[k]]
        per_img[m].append(float(c.per_class_iou[fg].mean()) if fg else float(c.per_class_iou[present].mean()))
per_img = {m: np.array(v) for m, v in per_img.items()}

score_df = pd.DataFrame(per_img, index=test_files)
print("mean per-image foreground IoU:\n", score_df.mean().round(4).to_string())
if len(avail) > 1:
    print("\npairwise correlation of per-image IoU:\n", score_df.corr().round(2).to_string())
    fig, ax = plt.subplots(figsize=(5, 4.5))
    a, b = avail[0], avail[1]
    ax.scatter(per_img[a], per_img[b], s=14, alpha=0.5)
    ax.plot([0, 1], [0, 1], "--", c="grey"); ax.set_xlabel(f"{a} IoU"); ax.set_ylabel(f"{b} IoU")
    ax.set_title("Per-image foreground IoU agreement"); plt.tight_layout(); plt.show()
allmin = np.min(np.stack(list(per_img.values())), axis=0)
hard = [test_files[i] for i in np.argsort(allmin)[:5]]
print("\nhardest images across ALL models:", [h[:18] for h in hard])

### 10.9 Verdict

In [ ]:
best_acc = max(models, key=lambda m: merged[m].get("foreground_mIoU", -1))
most_eff = min(models, key=lambda m: params_of(m))
# efficiency = foreground mIoU per million params
eff = {m: merged[m].get("foreground_mIoU", 0) / max(params_of(m), 1e-9) for m in models}
best_eff = max(eff, key=eff.get)
print(f"Best accuracy (foreground mIoU): {best_acc}  ({merged[best_acc].get('foreground_mIoU'):.4f})")
print(f"Smallest model                 : {most_eff}  ({params_of(most_eff)}M params)")
print(f"Best accuracy-per-param        : {best_eff}  ({eff[best_eff]:.4f} fg-mIoU/M)")
print(f"Hardest class across all models: light_hair_uv (thinnest, rarest, low contrast) — see per-class IoU")